# K-Nearest Neighbours (KNN)

## 1. Intuition

K-Nearest Neighbours (KNN) is an **instance-based**, **non-parametric** learning algorithm. Unlike linear regression or decision trees, KNN builds no explicit model during training — it simply stores the training data and defers all computation to prediction time.

When asked to classify a new point **x**, KNN:
1. Computes the distance from **x** to every training point.
2. Selects the *k* training points with the smallest distances.
3. Returns the **majority class** among those *k* neighbours (for classification) or their **mean target** (for regression).

The core intuition is: **similar inputs tend to have similar outputs**.

---

## 2. Why Non-Parametric?

A parametric model has a fixed number of learnable parameters (e.g. the slope and intercept in linear regression). KNN has **no parameters** to learn — the model is the dataset itself. Its complexity grows with the number of training samples, not with a fixed parameter budget.

---

## 3. Math

### Distance metric

The most common choice is **Euclidean distance** between two feature vectors **x**ᵢ, **x**ⱼ ∈ ℝᵖ:

$$
d(\mathbf{x}_i, \mathbf{x}_j) = \sqrt{\sum_{m=1}^{p}\left(x_{i,m} - x_{j,m}\right)^2}
$$

The **taxicab** (Manhattan) distance is an alternative:

$$
d_1(\mathbf{x}_i, \mathbf{x}_j) = \sum_{m=1}^{p}\left|x_{i,m} - x_{j,m}\right|
$$

Both are implemented in `rice_Ml.measures_ml.distances`.

### Classification rule

Let $\mathcal{N}_k(\mathbf{x})$ denote the index set of the *k* nearest training samples. The predicted class is:

$$
\hat{y} = \underset{c \in \mathcal{C}}{\arg\max}\; \sum_{i \in \mathcal{N}_k(\mathbf{x})} \mathbf{1}(y_i = c)
$$

where $\mathcal{C}$ is the set of classes and $\mathbf{1}(\cdot)$ is the indicator function.

### Regression rule (uniform weights)

$$
\hat{y} = \frac{1}{k}\sum_{i \in \mathcal{N}_k(\mathbf{x})} y_i
$$

With **inverse-distance weighting** (KNNRegressor option `weights="distance"`):

$$
\hat{y} = \frac{\sum_{i \in \mathcal{N}_k(\mathbf{x})} y_i / d_i}{\sum_{i \in \mathcal{N}_k(\mathbf{x})} 1 / d_i}
$$

---

## 4. The Role of *k* and Feature Scaling

- **Small *k*** → low bias, high variance. The decision boundary hugs individual training points (overfitting).
- **Large *k*** → high bias, low variance. The boundary smooths out, potentially underfitting.
- **Feature scaling is essential.** Features on a larger numeric scale dominate the distance computation. Always standardise (zero mean, unit variance) before fitting KNN.


---

## 5. Application — Pima Indians Diabetes Classification

We apply `KNNClassifier` to the [Pima Indians Diabetes Database](https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database) (768 patients, 8 clinical features, binary outcome).

| Feature | Description |
|---|---|
| Pregnancies | Number of pregnancies |
| Glucose | Plasma glucose (2-hr oral tolerance test) |
| BloodPressure | Diastolic blood pressure (mm Hg) |
| SkinThickness | Triceps skin fold thickness (mm) |
| Insulin | 2-hr serum insulin (μU/mL) |
| BMI | Body mass index (kg/m²) |
| DiabetesPedigreeFunction | Genetic risk score |
| Age | Age in years |

**Target:** `Outcome` — 1 = diabetes, 0 = no diabetes.


In [ ]:
import sys
from pathlib import Path

def _find_repo_root(marker: str = "pyproject.toml") -> Path | None:
    for candidate in [Path().resolve(), *Path().resolve().parents]:
        if (candidate / marker).exists():
            return candidate
    return None

REPO_ROOT = _find_repo_root()
if REPO_ROOT is not None:
    sys.path.insert(0, str(REPO_ROOT / "src"))
    print(f"Repo root: {REPO_ROOT}")
else:
    try:
        import rice_Ml as _check
        print(f"rice_Ml installed at: {Path(_check.__file__).resolve()}")
    except ImportError:
        raise ImportError(
            "rice_Ml not found. Install with:\n"
            "    pip install -e /path/to/cmor438-s2026-final-project"
        )

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import warnings
warnings.filterwarnings('ignore')

from rice_Ml.supervised_ml.knn.classifier import KNNClassifier
from rice_Ml.supervised_ml.knn.regressor import KNNRegressor
from rice_Ml.model_selection.split import train_test_split
from rice_Ml.metrics import accuracy

SEED = 42
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})
print("Imports OK")

---

## 6. Exploratory Data Analysis

In [ ]:
data = pd.read_csv("diabetes.csv")
print(data.shape)
data.head()

Zero values in Glucose, BloodPressure, SkinThickness, Insulin, and BMI are physiologically impossible — they represent missing data. We replace them with column medians.

In [ ]:
zero_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
for col in zero_cols:
    median_val = data.loc[data[col] != 0, col].median()
    data[col] = data[col].replace(0, median_val)

print("Class distribution:")
print(data["Outcome"].value_counts())
print(f"\nPositive rate: {data['Outcome'].mean():.1%}")

In [ ]:
colors = data["Outcome"].map({1: "#e74c3c", 0: "#3498db"})
legend_handles = [
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#e74c3c', markersize=8, label='Diabetes (1)'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#3498db', markersize=8, label='No Diabetes (0)'),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(data["Glucose"], data["BMI"], c=colors, alpha=0.4, s=20)
axes[0].set_xlabel("Glucose"); axes[0].set_ylabel("BMI")
axes[0].set_title("Glucose vs. BMI")
axes[0].legend(handles=legend_handles)

axes[1].scatter(data["Age"], data["DiabetesPedigreeFunction"], c=colors, alpha=0.4, s=20)
axes[1].set_xlabel("Age"); axes[1].set_ylabel("DiabetesPedigreeFunction")
axes[1].set_title("Age vs. Pedigree Function")
axes[1].legend(handles=legend_handles)

plt.tight_layout()
plt.show()

The Glucose vs. BMI plot shows that diabetic patients (red) cluster at higher glucose and moderately elevated BMI. There is considerable overlap — no single linear boundary separates the classes cleanly, which motivates a local, non-linear method like KNN.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
corr = data.corr()
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(corr.columns))); ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr.columns))); ax.set_yticklabels(corr.columns)
for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7)
ax.set_title("Feature Correlation Heatmap", fontweight="bold")
plt.tight_layout()
plt.show()

Glucose has the highest single-feature correlation with Outcome (≈ 0.49), followed by BMI and Age. Insulin and SkinThickness are moderately correlated with each other, which reflects their shared physiological relationship.

---

## 7. Decision Boundary — 2 Features (Glucose × BMI)

To visualise KNN's decision boundary, we first fit on just Glucose and BMI. We show how the boundary changes with *k*.

In [ ]:
def standardise(X_tr, X_te):
    mu, sigma = X_tr.mean(axis=0), X_tr.std(axis=0)
    return (X_tr - mu) / sigma, (X_te - mu) / sigma

FEATURE_COLS_2 = ["Glucose", "BMI"]
X2 = data[FEATURE_COLS_2].values
y  = data["Outcome"].values

X2_train, X2_test, y_train, y_test = train_test_split(
    X2, y, test_size=0.2, random_state=SEED, stratify=y
)
X2_train_s, X2_test_s = standardise(X2_train, X2_test)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

cmap_light = mcolors.ListedColormap(["#aec6e8", "#f5b8b0"])
cmap_bold  = mcolors.ListedColormap(["#3498db", "#e74c3c"])

for ax, k in zip(axes, [1, 9, 25]):
    clf = KNNClassifier(k=k).fit(X2_train_s, y_train)

    x_min, x_max = X2_train_s[:, 0].min() - 0.5, X2_train_s[:, 0].max() + 0.5
    y_min, y_max = X2_train_s[:, 1].min() - 0.5, X2_train_s[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.pcolormesh(xx, yy, Z, cmap=cmap_light, shading="auto", alpha=0.6)
    ax.scatter(X2_train_s[:, 0], X2_train_s[:, 1], c=y_train,
               cmap=cmap_bold, edgecolors="k", linewidths=0.3, s=18, alpha=0.7)
    acc = clf.score(X2_test_s, y_test)
    ax.set_title(f"k = {k}  (test acc = {acc:.2f})", fontweight="bold")
    ax.set_xlabel("Glucose (scaled)"); ax.set_ylabel("BMI (scaled)")

plt.suptitle("KNN Decision Boundary — Glucose × BMI", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

With k=1 the boundary is highly irregular — each training point carves out its own region (extreme overfitting). At k=9 the boundary smooths noticeably, and at k=25 it becomes nearly linear in this 2-D slice. The optimal k trades off these extremes.

---

## 8. Effect of Feature Scaling

KNN computes raw Euclidean distances. Glucose ranges ~50–200 while BMI ranges ~18–67. Without scaling, Glucose dominates the distance metric and BMI contributes almost nothing.

In [ ]:
k_val = 11

# Unscaled
clf_raw = KNNClassifier(k=k_val).fit(X2_train, y_train)
acc_raw = clf_raw.score(X2_test, y_test)

# Scaled
clf_scaled = KNNClassifier(k=k_val).fit(X2_train_s, y_train)
acc_scaled = clf_scaled.score(X2_test_s, y_test)

print(f"k={k_val}  Unscaled accuracy : {acc_raw:.3f}")
print(f"k={k_val}  Scaled accuracy   : {acc_scaled:.3f}")
print(f"Improvement from scaling: {acc_scaled - acc_raw:+.3f}")

Scaling consistently improves accuracy because it ensures all features contribute equally to the distance. **Always standardise before fitting KNN.**

---

## 9. Hyperparameter Tuning — Choosing *k*

In [ ]:
FEATURE_COLS_ALL = [c for c in data.columns if c != "Outcome"]
X_all = data[FEATURE_COLS_ALL].values

X_train, X_test, y_train_all, y_test_all = train_test_split(
    X_all, y, test_size=0.2, random_state=SEED, stratify=y
)
X_train_s, X_test_s = standardise(X_train, X_test)

k_range = range(1, 31)
train_accs, test_accs = [], []

for k in k_range:
    clf = KNNClassifier(k=k).fit(X_train_s, y_train_all)
    train_accs.append(clf.score(X_train_s, y_train_all))
    test_accs.append(clf.score(X_test_s, y_test_all))

best_k = k_range[int(np.argmax(test_accs))]
print(f"Best k on test set: {best_k}  (accuracy = {max(test_accs):.3f})")

plt.figure(figsize=(10, 4))
plt.plot(k_range, train_accs, label="Train accuracy", marker='o', markersize=4)
plt.plot(k_range, test_accs,  label="Test accuracy",  marker='s', markersize=4)
plt.axvline(best_k, color="grey", linestyle="--", label=f"Best k = {best_k}")
plt.xlabel("k"); plt.ylabel("Accuracy")
plt.title("KNN Accuracy vs. k — All Features (Scaled)", fontweight="bold")
plt.legend(); plt.tight_layout()
plt.show()

Train accuracy is 100% at k=1 (the model memorises every point) and falls monotonically as k increases. Test accuracy peaks in the k=9–15 range, confirming that a moderate neighbourhood size generalises best on this dataset.

---

## 10. Full-Feature Model — Confusion Matrix

In [ ]:
clf_best = KNNClassifier(k=best_k).fit(X_train_s, y_train_all)
y_pred = clf_best.predict(X_test_s)

print(f"Train accuracy: {clf_best.score(X_train_s, y_train_all):.3f}")
print(f"Test  accuracy: {accuracy(y_test_all, y_pred):.3f}")

cm = np.array([
    [((y_test_all==0) & (y_pred==0)).sum(), ((y_test_all==0) & (y_pred==1)).sum()],
    [((y_test_all==1) & (y_pred==0)).sum(), ((y_test_all==1) & (y_pred==1)).sum()],
])
class_names = ["No Diabetes", "Diabetes"]

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")
plt.colorbar(im, ax=ax)
ax.set_xticks(range(2)); ax.set_xticklabels(class_names)
ax.set_yticks(range(2)); ax.set_yticklabels(class_names)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title(f"Confusion Matrix — KNN (k={best_k})", fontweight="bold")
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm[i, j]}", ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.tight_layout()
plt.show()

The model correctly identifies most negative cases but has more false negatives (missed diabetics) than false positives, which is typical for imbalanced datasets with a 65/35 class split. In a clinical setting, false negatives are more costly than false positives, so a lower decision threshold (from `predict_proba`) would be appropriate.

---

## 11. Comparison to sklearn

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

sk_clf = KNeighborsClassifier(n_neighbors=best_k, metric="euclidean")
sk_clf.fit(X_train_s, y_train_all)
sk_pred = sk_clf.predict(X_test_s)

print(f"Our KNN  — train: {clf_best.score(X_train_s, y_train_all):.3f}  test: {accuracy(y_test_all, y_pred):.3f}")
print(f"sklearn  — train: {sk_clf.score(X_train_s, y_train_all):.3f}  test: {sk_clf.score(X_test_s, y_test_all):.3f}")

Both implementations should agree closely (within ~1%). Minor differences may arise from tie-breaking conventions in majority vote; our implementation breaks ties by selecting the lowest class label, matching sklearn's default behaviour.

---

## 12. Distance Metric Comparison — Euclidean vs. Taxicab

In [ ]:
results = {}
for k in [5, best_k, 21]:
    for metric in ["euclidean", "taxicab"]:
        clf = KNNClassifier(k=k, metric=metric).fit(X_train_s, y_train_all)
        results[(k, metric)] = clf.score(X_test_s, y_test_all)

print(f"{'k':>3}  {'metric':>12}  {'test acc':>10}")
print("-" * 30)
for (k, metric), acc_val in results.items():
    print(f"{k:>3}  {metric:>12}  {acc_val:>10.3f}")

On this dataset Euclidean and taxicab distances produce similar results after standardisation. After scaling, all features are on the same unit-variance scale, which reduces the geometric advantage of one metric over the other.

---

## 13. predict_proba and ROC Curve

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

proba = clf_best.predict_proba(X_test_s)[:, 1]
fpr, tpr, _ = roc_curve(y_test_all, proba)
auc = roc_auc_score(y_test_all, proba)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, lw=2, label=f"KNN (k={best_k})  AUC = {auc:.3f}")
plt.plot([0, 1], [0, 1], "k--", lw=1, label="Random baseline")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curve — KNN Classifier", fontweight="bold")
plt.legend()
plt.tight_layout()
plt.show()

print(f"AUC-ROC: {auc:.3f}")

An AUC around 0.82–0.85 indicates the classifier has good discriminative ability. The `predict_proba` output enables threshold tuning: lowering the threshold from 0.5 increases recall (fewer missed diabetics) at the cost of more false positives.